In [4]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [5]:
def generate_images(N=3000):

    X = np.zeros((N,8,8))
    y = np.zeros((N,1))

    for i in range(N):

        img = np.zeros((8,8))

        if np.random.rand() > 0.5:
            img[:,4] = 1
            y[i] = 0
        else:
            img[4,:] = 1
            y[i] = 1

        img += np.random.normal(0,0.1,(8,8))
        X[i] = img

    return X, y
X, y = generate_images(3000)

indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

train_end = int(0.7*len(X))
val_end = int(0.85*len(X))

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

print(X_train.shape, X_val.shape, X_test.shape)

(2100, 8, 8) (450, 8, 8) (450, 8, 8)


In [6]:
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

def sigmoid(z):
    return 1/(1+np.exp(-z))

def relu(z):
    return np.maximum(0,z)

def d_sigmoid(a):
    return a*(1-a)

def d_relu(z):
    return (z>0).astype(float)

def init_network(layers):
    params = {}
    for l in range(1,len(layers)):
        params["W"+str(l)] = np.random.randn(layers[l-1],layers[l])*0.1
        params["b"+str(l)] = np.zeros((1,layers[l]))
    return params

In [7]:
def forward(X,params):
    A = X
    cache = {}
    L = len(params)//2

    for l in range(1,L):
        Z = A @ params["W"+str(l)] + params["b"+str(l)]
        A = relu(Z)
        cache["A"+str(l)] = A
        cache["Z"+str(l)] = Z

    ZL = A @ params["W"+str(L)] + params["b"+str(L)]
    AL = sigmoid(ZL)

    cache["A"+str(L)] = AL
    return AL,cache

def compute_loss(y_hat,y):
    y_hat = np.clip(y_hat,1e-8,1-1e-8)
    return -np.mean(y*np.log(y_hat)+(1-y)*np.log(1-y_hat))

def compute_accuracy(y_hat,y):
    return np.mean((y_hat>=0.5)==y)



In [8]:
def backward(X,y,params,cache):
    grads = {}
    m = X.shape[0]
    L = len(params)//2

    dZ = cache["A"+str(L)] - y

    for l in reversed(range(1,L+1)):

        if l==1:
            A_prev = X
        else:
            A_prev = cache["A"+str(l-1)]

        grads["dW"+str(l)] = (A_prev.T @ dZ)/m
        grads["db"+str(l)] = np.sum(dZ,axis=0,keepdims=True)/m

        if l>1:
            dA_prev = dZ @ params["W"+str(l)].T
            dZ = dA_prev * d_relu(cache["Z"+str(l-1)])

    return grads

def update(params,grads,lr):
    for key in params:
        params[key] -= lr*grads["d"+key]
    return params


In [9]:
def train_dense():

    layers = [64,16,1]
    params = init_network(layers)

    for epoch in range(120):

        y_hat,cache = forward(X_train_flat,params)
        loss = compute_loss(y_hat,y_train)

        grads = backward(X_train_flat,y_train,params,cache)
        params = update(params,grads,0.01)

        if epoch%20==0:
            print("Epoch:",epoch,
                  "Loss:",round(loss,4),
                  "Acc:",round(compute_accuracy(y_hat,y_train),4))

    val_hat,_ = forward(X_val_flat,params)
    test_hat,_ = forward(X_test_flat,params)

    print("Validation Acc:",compute_accuracy(val_hat,y_val))
    print("Test Acc:",compute_accuracy(test_hat,y_test))

    return params

dense_params = train_dense()

Epoch: 0 Loss: 0.711 Acc: 0.4743
Epoch: 20 Loss: 0.6733 Acc: 0.5948
Epoch: 40 Loss: 0.6403 Acc: 0.9867
Epoch: 60 Loss: 0.6085 Acc: 1.0
Epoch: 80 Loss: 0.5744 Acc: 1.0
Epoch: 100 Loss: 0.5354 Acc: 1.0
Validation Acc: 1.0
Test Acc: 1.0


In [10]:
def init_cnn():
    params = {}
    params["K1"] = np.random.randn(3,3)*0.1
    params["b1"] = 0.0
    params["W2"] = np.random.randn(9,1)*0.1
    params["b2"] = 0.0
    return params

def conv_forward(X,K,b):
    N,H,W = X.shape
    F = K.shape[0]
    out = np.zeros((N,H-F+1,W-F+1))

    for n in range(N):
        for i in range(H-F+1):
            for j in range(W-F+1):
                region = X[n,i:i+F,j:j+F]
                out[n,i,j] = np.sum(region*K)+b
    return out


In [11]:
def maxpool_forward(X):
    N,H,W = X.shape
    out = np.zeros((N,H//2,W//2))

    for n in range(N):
        for i in range(H//2):
            for j in range(W//2):
                region = X[n,i*2:(i+1)*2,j*2:(j+1)*2]
                out[n,i,j] = np.max(region)
    return out

def cnn_forward(X,params):

    conv = conv_forward(X,params["K1"],params["b1"])
    relu = np.maximum(0,conv)
    pool = maxpool_forward(relu)

    flat = pool.reshape(len(pool),-1)
    Z = flat @ params["W2"] + params["b2"]
    out = sigmoid(Z)

    cache = (X,conv,relu,pool,flat,out)
    return out,cache


In [12]:
def maxpool_backward(dpool,relu):

    N,H,W = relu.shape
    d_relu = np.zeros_like(relu)

    for n in range(N):
        for i in range(H//2):
            for j in range(W//2):

                region = relu[n,i*2:(i+1)*2,j*2:(j+1)*2]
                max_val = np.max(region)

                for x in range(2):
                    for y in range(2):
                        if region[x,y]==max_val:
                            d_relu[n,i*2+x,j*2+y] = dpool[n,i,j]
    return d_relu

In [13]:
def conv_backward(dconv,X,K):

    N,H,W = X.shape
    F = K.shape[0]
    dK = np.zeros_like(K)

    for n in range(N):
        for i in range(H-F+1):
            for j in range(W-F+1):
                region = X[n,i:i+F,j:j+F]
                dK += region*dconv[n,i,j]
    return dK/N

def cnn_backward(y,cache,params):

    X,conv,relu,pool,flat,out = cache
    m = len(X)

    dout = out - y

    dW2 = flat.T @ dout / m
    db2 = np.sum(dout)/m

    dflat = dout @ params["W2"].T
    dpool = dflat.reshape(pool.shape)

    d_relu = maxpool_backward(dpool,relu)
    dconv = d_relu*(conv>0)

    dK = conv_backward(dconv,X,params["K1"])

    grads = {
        "dK1":dK,
        "db1":np.sum(dconv)/m,
        "dW2":dW2,
        "db2":db2
    }
    return grads

In [14]:
def train_cnn():

    params = init_cnn()

    for epoch in range(60):

        out,cache = cnn_forward(X_train,params)
        loss = compute_loss(out,y_train)

        grads = cnn_backward(y_train,cache,params)

        params["K1"] -= 0.01*grads["dK1"]
        params["b1"] -= 0.01*grads["db1"]
        params["W2"] -= 0.01*grads["dW2"]
        params["b2"] -= 0.01*grads["db2"]

        if epoch%10==0:
            print("Epoch:",epoch,
                  "Loss:",round(loss,4),
                  "Acc:",round(compute_accuracy(out,y_train),4))

    val_out,_ = cnn_forward(X_val,params)
    test_out,_ = cnn_forward(X_test,params)

    print("Validation Acc:",compute_accuracy(val_out,y_val))
    print("Test Acc:",compute_accuracy(test_out,y_test))

    return params

cnn_params = train_cnn()

Epoch: 0 Loss: 0.6894 Acc: 0.8481
Epoch: 10 Loss: 0.6884 Acc: 0.8667
Epoch: 20 Loss: 0.6874 Acc: 0.8833
Epoch: 30 Loss: 0.6862 Acc: 0.9062
Epoch: 40 Loss: 0.6849 Acc: 0.9219
Epoch: 50 Loss: 0.6835 Acc: 0.941
Validation Acc: 0.9666666666666667
Test Acc: 0.9733333333333334


PARAMETER COUNT
-----------------------------

# ----- Dense Baseline -----
# Architecture: 64 → 16 → 1

dense_input = 64
dense_hidden = 16
dense_output = 1

dense_params_layer1 = (dense_input * dense_hidden) + dense_hidden
dense_params_layer2 = (dense_hidden * dense_output) + dense_output

total_dense_params = dense_params_layer1 + dense_params_layer2

print("Dense Layer 1 Params:", dense_params_layer1)
print("Dense Layer 2 Params:", dense_params_layer2)
print("Total Dense Parameters:", total_dense_params)

conv_filter_size = 3
conv_in_channels = 1
conv_out_channels = 1

conv_params = (conv_filter_size * conv_filter_size * conv_in_channels) + 1

flatten_size = 9
dense_output = 1

cnn_dense_params = (flatten_size * dense_output) + 1

total_cnn_params = conv_params + cnn_dense_params

print("Conv Layer Params:", conv_params)
print("Dense Output Params:", cnn_dense_params)
print("Total CNN Parameters:", total_cnn_params)


print("Total Dense Parameters:", total_dense_params)
print("Total CNN Parameters:", total_cnn_params)

if total_dense_params > total_cnn_params:
    print("\nCNN uses fewer parameters than Dense.")
else:
    print("\nDense uses fewer parameters than CNN.")




Results:
Dense Layer 1 Params: 1040
Dense Layer 2 Params: 17
Total Dense Parameters: 1057

---------------------------

Conv Layer Params: 10
Dense Output Params: 10
Total CNN Parameters: 20

FINAL COMPARISON
==============================
Total Dense Parameters: 1057
Total CNN Parameters: 20

CNN uses fewer parameters than Dense.